In [1]:
import optuna
import gymnasium as gym
import numpy as np
import pandas as pd
from stable_baselines3 import A2C
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Load Tesla stock data
data = pd.read_csv('tesla.csv', parse_dates=['Date'], index_col='Date')

In [3]:
# Convert 'Volume' column to numeric by removing commas
data['Volume'] = data['Volume'].replace(',', '', regex=True).astype(float)

In [4]:
# Feature Engineering
data['SMA_10'] = data['Close'].rolling(window=10).mean()
data['EMA_10'] = data['Close'].ewm(span=10, adjust=False).mean()
data['Daily_Return'] = data['Close'].pct_change()
data['Volatility'] = data['Daily_Return'].rolling(window=10).std()

In [5]:

def compute_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

data['RSI'] = compute_rsi(data['Close'])
data.dropna(inplace=True)

In [6]:
# Normalize Data
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(data[['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'EMA_10', 'Volatility', 'RSI']])
scaled_data = pd.DataFrame(scaled_features, columns=['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'EMA_10', 'Volatility', 'RSI'])

In [7]:
# Define custom TradingEnv
class TradingEnv(gym.Env):
    def __init__(self, data, initial_balance=10000):
        super(TradingEnv, self).__init__()
        self.data = data
        self.initial_balance = initial_balance
        self.current_step = 0
        self.balance = initial_balance
        self.position = 0  # No position initially
        self.previous_price = 0
        self.action_space = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Box(low=0, high=1, shape=(self.data.shape[1],), dtype=np.float32)

    def reset(self, seed=None, options=None):
        self.current_step = 0
        self.balance = self.initial_balance
        self.position = 0
        self.previous_price = 0
        return self.data.iloc[self.current_step].values, {}

    def step(self, action):
        current_price = self.data['Close'].iloc[self.current_step]
        reward = 0
        if action == 1:  # Buy
            if self.position == 0:
                self.position = 1
                self.previous_price = current_price
        elif action == 2:  # Sell
            if self.position == 1:
                reward = current_price - self.previous_price
                self.position = 0
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        reward += self.position * (current_price - self.previous_price)
        obs = self.data.iloc[self.current_step].values
        return obs, reward, done, {}, {}

    def render(self):
        print(f'Step: {self.current_step}, Balance: {self.balance}, Position: {self.position}')

# Hyperparameter tuning with Optuna
def objective(trial):
    env = DummyVecEnv([lambda: TradingEnv(scaled_data)])
    env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    gamma = trial.suggest_float('gamma', 0.95, 0.99)
    ent_coef = trial.suggest_float('ent_coef', 0.0, 0.1)
    vf_coef = trial.suggest_float('vf_coef', 0.25, 0.75)
    n_steps = trial.suggest_int('n_steps', 5, 50)
    model = A2C("MlpPolicy", env, learning_rate=learning_rate, gamma=gamma,
                ent_coef=ent_coef, vf_coef=vf_coef, n_steps=n_steps, verbose=0)
    model.learn(total_timesteps=50000)
    total_reward = 0
    obs = env.reset()
    for _ in range(1000):
        action, _ = model.predict(obs)
        obs, reward, done, _ = env.step(action)
        total_reward += reward
        if done:
            break
    return total_reward

In [8]:
# Hyperparameter tuning with Optuna
def objective(trial):
    env = DummyVecEnv([lambda: TradingEnv(scaled_data)])
    env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    gamma = trial.suggest_float('gamma', 0.95, 0.99)
    ent_coef = trial.suggest_float('ent_coef', 0.0, 0.1)
    vf_coef = trial.suggest_float('vf_coef', 0.25, 0.75)
    n_steps = trial.suggest_int('n_steps', 5, 50)
    model = A2C("MlpPolicy", env, learning_rate=learning_rate, gamma=gamma,
                ent_coef=ent_coef, vf_coef=vf_coef, n_steps=n_steps, verbose=0)
    model.learn(total_timesteps=50000)
    total_reward = 0
    obs = env.reset()
    for _ in range(1000):
        action, _ = model.predict(obs)
        obs, reward, done, _ = env.step(action)
        total_reward += reward
        if done:
            break
    return total_reward

In [ ]:
# Run Optuna optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2025-02-13 12:38:15,000] A new study created in memory with name: no-name-c09e2979-6869-4f5a-ae9c-923c24d33323
C:\Users\goel\AppData\Local\Temp\ipykernel_1620\2309285215.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
C:\Users\goel\anaconda3\ANACONDA\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might tak